In [ ]:
# ============================================================
# 05_gold_analytics.ipynb
#
# Purpose:
# Call the reusable analytics-serving production module,
# verify its results, and inspect the analytical SQL views.
# ============================================================

from pathlib import Path
from pprint import pprint
import importlib
import sys

import duckdb
from IPython.display import display


# ============================================================
# 1. Locate the project root
# ============================================================

current_directory = Path.cwd().resolve()

if (current_directory / "src").exists():
    PROJECT_ROOT = current_directory

elif (current_directory.parent / "src").exists():
    PROJECT_ROOT = current_directory.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "the src directory."
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ============================================================
# 2. Import the reusable production module
# ============================================================

import src.analytics_serving as analytics_serving

# Reloading ensures the notebook uses the newest saved version.
importlib.reload(analytics_serving)

build_analytics_serving_layer = (
    analytics_serving.build_analytics_serving_layer
)


# ============================================================
# 3. Define input and output paths
# ============================================================

DATA_PATH = PROJECT_ROOT / "data"
GOLD_PATH = DATA_PATH / "gold"
ANALYTICS_PATH = DATA_PATH / "analytics"

merchant_input_path = (
    GOLD_PATH / "dim_merchant.parquet"
)

date_input_path = (
    GOLD_PATH / "dim_date.parquet"
)

transaction_input_path = (
    GOLD_PATH / "fact_transaction.parquet"
)

analytics_database_path = (
    ANALYTICS_PATH
    / "banking_analytics.duckdb"
)


print("Project root:", PROJECT_ROOT)

print(
    "Merchant input:",
    merchant_input_path,
)

print(
    "Date input:",
    date_input_path,
)

print(
    "Transaction input:",
    transaction_input_path,
)

print(
    "Analytics database:",
    analytics_database_path,
)


# ============================================================
# 4. Build the analytics serving layer
# ============================================================

analytics_result = build_analytics_serving_layer(
    merchant_input_path=merchant_input_path,
    date_input_path=date_input_path,
    transaction_input_path=transaction_input_path,
    analytics_database_path=analytics_database_path,
)


print("\nAnalytics serving-layer execution result:\n")

pprint(analytics_result)


# ============================================================
# 5. Validate execution metrics
# ============================================================

assert analytics_result["status"] == "SUCCESS"

assert analytics_database_path.exists()

assert analytics_result[
    "validation"
]["is_valid"]

assert analytics_result[
    "validation"
]["merchant_rows"] > 0

assert analytics_result[
    "validation"
]["date_rows"] > 0

assert analytics_result[
    "validation"
]["transaction_rows"] > 0

assert analytics_result[
    "validation"
]["duplicate_merchant_keys"] == 0

assert analytics_result[
    "validation"
]["duplicate_date_keys"] == 0

assert analytics_result[
    "validation"
]["duplicate_transaction_keys"] == 0

assert analytics_result[
    "validation"
]["missing_merchant_keys"] == 0

assert analytics_result[
    "validation"
]["missing_date_keys"] == 0

assert analytics_result[
    "validation"
]["missing_transaction_keys"] == 0

assert analytics_result[
    "validation"
]["invalid_merchant_foreign_keys"] == 0

assert analytics_result[
    "validation"
]["invalid_date_foreign_keys"] == 0


print(
    "\nAnalytics execution-metric validation passed."
)


# ============================================================
# 6. Display the platform-level KPI result
# ============================================================

print("\nPlatform-level KPI metrics:")

platform_kpi_result = analytics_result[
    "platform_kpis"
]

pprint(platform_kpi_result)


assert (
    platform_kpi_result[
        "total_transactions"
    ]
    == analytics_result[
        "validation"
    ]["transaction_rows"]
)


# ============================================================
# 7. Open the analytics database for inspection
# ============================================================

inspection_connection = duckdb.connect(
    str(analytics_database_path),
    read_only=True,
)


try:
    # ========================================================
    # 8. Confirm that all expected SQL views exist
    # ========================================================

    available_views_df = (
        inspection_connection.execute(
            """
            SELECT
                table_name AS view_name

            FROM information_schema.views

            WHERE table_schema = 'main'

            ORDER BY
                table_name
            """
        ).fetchdf()
    )


    print("\nAvailable analytics views:")

    display(available_views_df)


    expected_views = set(
        analytics_result["views_created"]
    )

    available_views = set(
        available_views_df["view_name"]
    )

    assert expected_views.issubset(
        available_views
    )


    print("Analytics view verification passed.")


    # ========================================================
    # 9. Query platform KPIs
    # ========================================================

    platform_kpi_df = (
        inspection_connection.execute(
            """
            SELECT *
            FROM vw_platform_kpis
            """
        ).fetchdf()
    )


    print("\nPLATFORM KPIs")

    display(platform_kpi_df)


    # ========================================================
    # 10. Query transaction-method performance
    # ========================================================

    transaction_method_df = (
        inspection_connection.execute(
            """
            SELECT *
            FROM vw_transaction_method_summary

            ORDER BY
                transaction_count DESC
            """
        ).fetchdf()
    )


    print("\nTRANSACTIONS BY METHOD")

    display(transaction_method_df)


    assert (
        transaction_method_df[
            "transaction_count"
        ].sum()
        == platform_kpi_result[
            "total_transactions"
        ]
    )


    # ========================================================
    # 11. Query annual transaction performance
    # ========================================================

    annual_transaction_df = (
        inspection_connection.execute(
            """
            SELECT *
            FROM vw_annual_transaction_summary

            ORDER BY
                calendar_year
            """
        ).fetchdf()
    )


    print("\nANNUAL TRANSACTION ACTIVITY")

    display(annual_transaction_df)


    assert (
        annual_transaction_df[
            "transaction_count"
        ].sum()
        == platform_kpi_result[
            "total_transactions"
        ]
    )


    print(
        "\nAnalytical view reconciliation passed."
    )


finally:
    inspection_connection.close()

    print(
        "DuckDB inspection connection closed."
    )


# ============================================================
# 12. Final serving-layer summary
# ============================================================

print("\n" + "=" * 70)

print(
    "ANALYTICS SERVING LAYER COMPLETED SUCCESSFULLY"
)

print("=" * 70)

print(
    "Database:",
    analytics_result[
        "analytics_database_path"
    ],
)

print(
    "Views created:",
    len(
        analytics_result[
            "views_created"
        ]
    ),
)

print(
    "Merchant rows:",
    analytics_result[
        "validation"
    ]["merchant_rows"],
)

print(
    "Date rows:",
    analytics_result[
        "validation"
    ]["date_rows"],
)

print(
    "Transaction rows:",
    analytics_result[
        "validation"
    ]["transaction_rows"],
)

print(
    "Total transaction amount:",
    platform_kpi_result[
        "total_transaction_amount"
    ],
)

print(
    "Fraudulent transactions:",
    platform_kpi_result[
        "fraudulent_transactions"
    ],
)

print(
    "Fraud transaction rate:",
    str(
        platform_kpi_result[
            "fraud_transaction_rate_percent"
        ]
    )
    + "%",
)

print(
    "Validation passed:",
    analytics_result[
        "validation"
    ]["is_valid"],
)

print("=" * 70)